# USTC 东校区 LES：项目起点

使用 **turbpy** 内核。本 Notebook 做参数预检查和地转驱动力校验，尚不计算校园 LES 风场。
已获取 OSM 建筑轮廓；高度含情景假设，气象条件待核实。草案参数仅用于规划。

In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if not (ROOT / 'configs').is_dir():
    ROOT = ROOT.parent
assert (ROOT / 'configs/campus_draft.json').is_file(), '请从项目或 notebooks 目录启动'
sys.path.insert(0, str(ROOT / 'src'))
from ustc_les.planning import read_config, preflight
from ustc_les.physics import coriolis_parameter, inertial_solution
print('Python:', sys.executable)
assert Path(sys.prefix).name.lower() == 'turbpy', '请选择 turbpy 内核'


In [ ]:
config = read_config(ROOT / 'configs/campus_draft.json')
report = preflight(config, ROOT)
print(json.dumps(report, ensure_ascii=False, indent=2))


## 地转驱动力的独立物理检查

下图是均匀、无摩擦流体相对地转风的惯性振荡解析解，用于检查旋转符号。
它不包含湍流、建筑、热力或壁面，因此不是校园风场。

In [ ]:
f = coriolis_parameter(config['latitude_deg'])
ug, vg = config['geostrophic_velocity_m_s']
t = np.linspace(0, 2 * np.pi / abs(f), 300)
u, v = inertial_solution(t, ug + 2, vg + 1, ug, vg, f)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(t / 3600, u, label='u (east)')
axes[0].plot(t / 3600, v, label='v (north)')
axes[0].set(xlabel='Time (h)', ylabel='Velocity (m/s)', title='Inviscid forcing check')
axes[0].legend()
axes[1].plot(u - ug, v - vg)
axes[1].plot(u[0] - ug, v[0] - vg, 'o', label='Start')
axes[1].set(xlabel='u - Ug (m/s)', ylabel='v - Vg (m/s)', title='Inertial orbit (not LES)')
axes[1].set_aspect('equal', adjustable='box')
axes[1].legend()
fig.tight_layout()
plt.show()


## 下一步

按 `docs/geometry.md` 获取校园边界与建筑清单，核实高度及坐标；按 `docs/methodology.md` 选择并验证三维 LES 求解器。
参考：[JHTDB SABL](https://turbulence.idies.jhu.edu/datasets/geophysicalTurbulence/sabl)。